In [3]:
import cv2
import numpy as np
import keras
from keras import Model
from keras.layers import Dense
from keras.optimizers import Adam
from keras.applications.vgg16 import VGG16
from keras.models import model_from_json

img_path = "image.png"
model_json_path = "model.json"
model_weights_path = "model.h5"

img = cv2.imread(img_path)

cv2.setUseOptimized(True)
ss = cv2.ximgproc.segmentation.createSelectiveSearchSegmentation()
ss.setBaseImage(img)
ss.switchToSelectiveSearchFast()
rects = ss.process()

try:
    with open(model_json_path, "r", encoding="utf-8") as f:
        loaded_model_json = f.read()
    model_final = model_from_json(loaded_model_json)
    model_final.load_weights(model_weights_path)
except:
    vggmodel = VGG16(weights="imagenet", include_top=True)
    for layer in vggmodel.layers[:15]:
        layer.trainable = False
    X = vggmodel.layers[-2].output
    predictions = Dense(2, activation="softmax")(X)
    model_final = Model(inputs=vggmodel.input, outputs=predictions)
    model_final.load_weights(model_weights_path)

opt = Adam(learning_rate=0.0001)
model_final.compile(
    loss=keras.losses.categorical_crossentropy, optimizer=opt, metrics=["accuracy"]
)

detections = []
for rect in rects[:10000]:
    x, y, w, h = rect
    if w <= 0 or h <= 0:
        continue
    roi = img[y : y + h, x : x + w]
    if roi.size == 0:
        continue
    resized = cv2.resize(roi, (224, 224), interpolation=cv2.INTER_AREA)
    batch = np.expand_dims(resized, axis=0)
    out = model_final.predict(batch, verbose=0)
    score = float(out[0][0])
    if score > 0.65:
        detections.append((score, x, y, w, h))

best = max(detections, key=lambda z: z[0])
worst = min(detections, key=lambda z: z[0])

print("count =", len(detections))
print("best_score =", round(best[0], 3))
print("best_x =", int(best[1]))
print("best_y =", int(best[2]))
print("best_w =", int(best[3]))
print("best_h =", int(best[4]))
print("worst_score =", round(worst[0], 3))
print("worst_x =", int(worst[1]))
print("worst_y =", int(worst[2]))
print("worst_w =", int(worst[3]))
print("worst_h =", int(worst[4]))

print("\nALL DETECTIONS SORTED:")
for d in sorted(detections, key=lambda z: z[0]):
    print(round(d[0], 3), int(d[1]), int(d[2]), int(d[3]), int(d[4]))


count = 9
best_score = 0.915
best_x = 83
best_y = 71
best_w = 34
best_h = 27
worst_score = 0.662
worst_x = 78
worst_y = 71
worst_w = 36
worst_h = 46

ALL DETECTIONS SORTED:
0.662 78 71 36 46
0.76 89 71 25 27
0.805 226 155 30 33
0.808 78 71 36 26
0.813 90 71 24 26
0.844 223 153 33 35
0.849 82 71 32 27
0.857 222 154 34 34
0.915 83 71 34 27
